In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os
import sys
sys.path.insert(0, '..')

import torch
import numpy as np
import matplotlib.pyplot as plt
from utils import data

device = torch.device('cuda')
print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
# ===== Configuration =====
mouse_id    = 3
data_path   = '../data'
weight_path = './checkpoints/vit_frozen'
os.makedirs(weight_path, exist_ok=True)

import os

HF_TOKEN = os.getenv("HF_TOKEN")                
MODEL_NAME = 'facebook/dinov3-vitb16-pretrain-lvd1689m' # 'facebook/dinov3-vits16-pretrain-lvd1689m' # ViT-Small, ~21M params

# --- Token & layer config ---
TOKEN_TYPE     = 'patch'   # 'patch': spatial readout  |  'cls': global/linear readout
EXTRACT_LAYERS = [3]      # None = last layer; e.g. [8,9,10,11] = last 4 of 12 blocks

# --- Training config ---
MAX_EPOCHS = 200   # hard upper limit; early stopping will usually kick in sooner
PATIENCE   = 5     # epochs without varexp improvement → reduce LR (at patience//2), stop (at patience)

np.random.seed(1)

In [ ]:
# Load full 66×264 images WITHOUT normalization.
# We normalize AFTER spatial preprocessing so the z-score stats are
# computed on the same pixels the model will actually see.
img = data.load_images(data_path, mouse_id, file=data.img_file_name[mouse_id],
                       crop=False, normalize=False)
print('raw img shape:', img.shape, '  dtype:', img.dtype,
      '  range: [%.1f, %.1f]' % (img.min(), img.max()))

In [ ]:
# ===== Image preprocessing: 66×264 → (N, 3, 32, 64), ImageNet-normalized =====
# Pipeline:
#   1. Resize  66×264  →  64×256  (bilinear)
#   2. Crop left half  →  64×128
#   3. Downsample 2×   →  32×64   (bilinear)
#   4. Per-image [0, 1] normalization
#   5. Repeat to 3 channels (grayscale → pseudo-RGB)
#   6. ImageNet normalize: mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]
from utils.vit_encoding_model import preprocess_images

img = preprocess_images(img, batch_size=2000)

print('preprocessed img shape:', img.shape)
print('  mean=%.4f  std=%.4f  range=[%.2f, %.2f]' % (img.mean(), img.std(), img.min(), img.max()))

In [ ]:
# Load neurons
fname = '%s_nat60k_%s.npz' % (data.db[mouse_id]['mname'], data.db[mouse_id]['datexp'])
spks, istim_train, istim_test, xpos, ypos, spks_rep_all = data.load_neurons(
    file_path=os.path.join(data_path, fname), mouse_id=mouse_id
)
n_stim, n_neurons = spks.shape

In [ ]:
# Split train / validation
itrain, ival = data.split_train_val(istim_train, train_frac=0.9)

In [ ]:
# Normalize neural data
spks, spks_rep_all = data.normalize_spks(spks, spks_rep_all, itrain)
print('spks shape:', spks.shape)
print('spks_rep_all shape:', spks_rep_all.shape)

In [ ]:
# Prepare tensors
ineur = np.arange(0, n_neurons)
spks_train = torch.from_numpy(spks[itrain][:, ineur])
spks_val   = torch.from_numpy(spks[ival][:, ineur])

# Images: (N, 3, 32, 64) already from preprocess_images()
img_train = torch.from_numpy(img[istim_train][itrain]).to(device)
img_val   = torch.from_numpy(img[istim_train][ival]).to(device)
img_test  = torch.from_numpy(img[istim_test]).to(device)

print('spks_train:', spks_train.shape, spks_train.min().item(), spks_train.max().item())
print('spks_val:  ', spks_val.shape, spks_val.min().item(), spks_val.max().item())
print('img_train: ', img_train.shape, img_train.min().item(), img_train.max().item())
print('img_test:  ', img_test.shape)

In [ ]:
# ===== Build ViT model (frozen backbone) =====
from utils.vit_encoding_model import build_vit_encoder, make_model_name

# vit_input_size=(112, 224): ViT internal input; aspect ratio 0.5 matches 32×64.
#   Gives 7×14 patch grid (patch_size=16).
# out_spatial_size=(16, 32): upsample patch features to 2× downsampled model input,
#   matching the CNN convention (pool 2× from 32×64 → 16×32).
#   Set to None to use the native 7×14 patch grid instead.
# token_type: 'patch' uses spatial patch tokens; 'cls' uses the global CLS token.

model = build_vit_encoder(
    n_neurons        = len(ineur),
    model_name       = MODEL_NAME,
    vit_input_size   = (64, 128),   # ViT internal resize (H, W), divisible by 16
    out_spatial_size = (4, 8),     # readout resolution (half of 32×64 model input)
    extract_layers   = EXTRACT_LAYERS,
    freeze_backbone  = True,
    poisson          = True,
    hf_token         = HF_TOKEN,
    device           = device,
)

In [ ]:
# ===== Train readout (frozen backbone) =====
from utils.vit_encoding_model import train_readout

# Model name encodes: mouse, date, ViT variant, token type, layers used
model_filename = make_model_name(
    data.mouse_names[mouse_id], data.exp_date[mouse_id],
    MODEL_NAME, TOKEN_TYPE, EXTRACT_LAYERS
)
model_path = os.path.join(weight_path, model_filename)
print('Checkpoint:', model_path)

if not os.path.exists(model_path):
    best_state = train_readout(
        model,
        spks_train, spks_val,
        img_train,  img_val,
        max_epochs = MAX_EPOCHS,
        batch_size = 64,
        lr         = 1e-3,
        l2_readout = 0.1,
        clamp      = True,
        patience   = PATIENCE,
        device     = device,
    )
    torch.save(best_state, model_path)
    print('Saved:', model_path)
else:
    model.load_state_dict(torch.load(model_path, map_location=device))
    print('Loaded:', model_path)

In [ ]:
# ===== Evaluate: FEVE on test set =====
from utils import metrics, model_trainer
# ViT model predictions
model.eval()
n_test   = img_test.shape[0]
vit_pred = []
with torch.no_grad():
    for k in range(0, n_test, 64):
        pred = model(img_test[k:k+64])
        vit_pred.append(pred.cpu().numpy())
vit_pred = np.vstack(vit_pred)
print('vit_pred:', vit_pred.shape, vit_pred.min(), vit_pred.max())

In [ ]:
# FEVE metric
vit_fev, vit_feve = metrics.feve(spks_rep_all, vit_pred)

threshold = 0.15
valid = np.where(vit_fev > threshold)[0]
print(f'Valid neurons (FEV > {threshold}): {len(valid)} / {len(vit_fev)}')
print(f'FEVE (test, ViT frozen): {np.mean(vit_feve[vit_fev > threshold]):.4f}')

In [ ]:
plt.scatter(xpos , ypos, c=vit_feve, cmap='viridis', s=1, vmin=0, vmax=0.5)
plt.colorbar(label='FEVE (ViT frozen)')
plt.title(f'Mouse {data.mouse_names[mouse_id]} - ViT Frozen')
plt.xlabel('X pos')
plt.ylabel('Y pos')
plt.show()  